```
urbansound8k/
├── features.pkl   -- {sample_id: {"audio_path": <path_relative_to_dataset_root>}}
├── targets.pkl    -- {sample_id: {"class_id": <int>}}
├── train_keys.pkl -- {idx: {"key": sample_id}, ...}
├── val_keys.pkl   -- {idx: {"key": sample_id}, ...}
└── test_keys.pkl  -- {idx: {"key": sample_id}, ...}
```

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import soundfile as sf
from datasets import load_dataset
from tqdm.auto import tqdm

## Setup

In [ ]:
data_dir = Path(".").resolve().parent / "data"
raw_dir = data_dir / "raw" / "UrbanSound8K"
preprocessed_dir = data_dir / "preprocessed" / "urbansound8k"

val_folds = [9]
test_folds = [10]

print(f"Raw audio dir: {raw_dir}")
print(f"Preprocessed dir: {preprocessed_dir}")
print(f"Val folds: {val_folds}, Test folds: {test_folds}")

## Download

In [ ]:
ds = load_dataset("danavery/urbansound8K", split="train")
print(f"Loaded {len(ds)} samples")
print(f"Features: {ds.features}")

In [ ]:
raw_dir.mkdir(parents=True, exist_ok=True)

for sample in tqdm(ds, desc="Saving audio files"):
    fold = sample["fold"]
    filename = sample["slice_file_name"]
    audio = sample["audio"]

    fold_dir = raw_dir / "audio" / f"fold{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    audio_path = fold_dir / filename
    if not audio_path.exists():
        sf.write(str(audio_path), audio["array"], audio["sampling_rate"])

## Build

In [ ]:
metadata = ds.to_pandas()[
    [
        "slice_file_name", 
        "fsID", 
        "start", 
        "end", 
        "salience", 
        "fold", 
        "classID", 
        "class"
    ]
]

print(f"Total samples: {len(metadata)}")
print(f"Classes:")
metadata["class"].value_counts().sort_index()

In [ ]:
features = {}
targets = {}

for _, row in metadata.iterrows():
    sample_id = row["slice_file_name"].replace(".wav", "")
    fold = row["fold"]
    audio_path = str(Path("audio") / f"fold{fold}" / row["slice_file_name"])

    features[sample_id] = {"audio_path": audio_path}
    targets[sample_id] = {"class_id": int(row["classID"])}

print(f"Features: {len(features)} samples")
print(f"Targets:  {len(targets)} samples")

### train / val / test split

In [ ]:
train_keys = {}
val_keys = {}
test_keys = {}

train_idx = val_idx = test_idx = 0

for _, row in metadata.iterrows():
    sample_id = row["slice_file_name"].replace(".wav", "")
    fold = row["fold"]

    if fold in test_folds:
        test_keys[test_idx] = {"key": sample_id}
        test_idx += 1
    elif fold in val_folds:
        val_keys[val_idx] = {"key": sample_id}
        val_idx += 1
    else:
        train_keys[train_idx] = {"key": sample_id}
        train_idx += 1

all_folds = set(metadata["fold"].unique().tolist())
train_folds = all_folds - set(val_folds) - set(test_folds)

print(f"Train folds: {sorted(train_folds)} -> {len(train_keys)} samples")
print(f"Val folds:   {val_folds} -> {len(val_keys)} samples")
print(f"Test folds:  {test_folds} -> {len(test_keys)} samples")

## Save preprocessed

In [ ]:
preprocessed_dir.mkdir(parents=True, exist_ok=True)

for name, data in [
    ("features", features),
    ("targets", targets),
    ("train_keys", train_keys),
    ("val_keys", val_keys),
    ("test_keys", test_keys),
]:
    path = preprocessed_dir / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"Saved {name}: {len(data)} entries -> {path}")

## EDA

Анализ ниже использует уже сохранённые локальные артефакты из `data/preprocessed/urbansound8k` и аудиофайлы из `data/raw/UrbanSound8K`. Его можно запускать независимо от этапов загрузки и предварительной обработки выше.

In [ ]:
eda_preprocessed_dir = data_dir / "preprocessed" / "urbansound8k"
eda_raw_dir = data_dir / "raw" / "UrbanSound8K"

class_names = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music",
}

with open(eda_preprocessed_dir / "features.pkl", "rb") as f:
    eda_features = pickle.load(f)
with open(eda_preprocessed_dir / "targets.pkl", "rb") as f:
    eda_targets = pickle.load(f)
with open(eda_preprocessed_dir / "train_keys.pkl", "rb") as f:
    eda_train_keys = pickle.load(f)
with open(eda_preprocessed_dir / "val_keys.pkl", "rb") as f:
    eda_val_keys = pickle.load(f)
with open(eda_preprocessed_dir / "test_keys.pkl", "rb") as f:
    eda_test_keys = pickle.load(f)

print(f"EDA features: {len(eda_features)}")
print(f"EDA targets: {len(eda_targets)}")
print(f"EDA train keys: {len(eda_train_keys)}")
print(f"EDA val keys: {len(eda_val_keys)}")
print(f"EDA test keys: {len(eda_test_keys)}")

In [ ]:
def resolve_eda_audio_path(audio_path: str) -> Path:
    path = Path(audio_path)
    if path.exists():
        return path
    if "audio" in path.parts:
        audio_idx = path.parts.index("audio")
        candidate = eda_raw_dir.joinpath(*path.parts[audio_idx:])
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Audio file not found: {audio_path}")

eda_split_by_key = {}
eda_split_by_key.update({payload["key"]: "train" for payload in eda_train_keys.values()})
eda_split_by_key.update({payload["key"]: "val" for payload in eda_val_keys.values()})
eda_split_by_key.update({payload["key"]: "test" for payload in eda_test_keys.values()})

eda_rows = []
for sample_id, feature in tqdm(eda_features.items(), desc="Building EDA metadata"):
    audio_file = resolve_eda_audio_path(feature["audio_path"])
    info = sf.info(audio_file)
    fold_name = next(part for part in audio_file.parts if part.startswith("fold"))
    class_id = int(eda_targets[sample_id]["class_id"])
    eda_rows.append(
        {
            "sample_id": sample_id,
            "audio_path": str(audio_file),
            "fold": int(fold_name.replace("fold", "")),
            "split": eda_split_by_key[sample_id],
            "class_id": class_id,
            "class": class_names[class_id],
            "duration": info.duration,
            "sampling_rate": info.samplerate,
            "channels": info.channels,
            "frames": info.frames,
        }
    )

eda_metadata = pd.DataFrame(eda_rows).sort_values(["split", "class", "sample_id"]).reset_index(drop=True)

display(eda_metadata.head())
display(eda_metadata[["duration", "sampling_rate", "channels", "frames"]].describe().transpose())

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

class_counts = (
    eda_metadata["class"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="num_samples")
    .sort_values("num_samples", ascending=False)
)

split_counts = (
    eda_metadata["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="num_samples")
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(
    data=class_counts,
    x="num_samples",
    y="class",
    hue="class",
    palette="viridis",
    ax=axes[0],
    legend=False  # disables legend which is redundant for y variable as hue
)
axes[0].set_title("Samples per Class")
axes[0].set_xlabel("Number of samples")
axes[0].set_ylabel("Class")

sns.barplot(
    data=split_counts,
    x="split",
    y="num_samples",
    hue="split",
    palette="Set2",
    ax=axes[1],
    legend=False  # disables legend which is redundant for x variable as hue
)
axes[1].set_title("Samples per Split")
axes[1].set_xlabel("Split")
axes[1].set_ylabel("Number of samples")

plt.tight_layout()
plt.show()

### Заметки о сбалансированности
Распределение по классам близко к равномерному, но не является полностью однородным. Редкие классы, такие как `gun_shot` и `car_horn`, всё равно следует отслеживать отдельно при интерпретации качества модели.

In [ ]:
fold_class_counts = (
    eda_metadata.groupby(["fold", "class"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

plt.figure(figsize=(16, 8))
sns.heatmap(fold_class_counts, cmap="mako", linewidths=0.5)
plt.title("Class Distribution by Fold")
plt.xlabel("Class")
plt.ylabel("Fold")
plt.tight_layout()
plt.show()

### Заметки о фолдах
Официальные фолды не являются полностью однородными по составу классов. Для экспериментов предопределённый протокол разбиения на фолды остаётся более надёжным, чем случайное разделение.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(data=eda_metadata, x="duration", bins=30, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Clip Duration Distribution")
axes[0].set_xlabel("Duration, seconds")
axes[0].set_ylabel("Number of samples")

median_order = (
    eda_metadata.groupby("class")["duration"]
    .median()
    .sort_values(ascending=False)
    .index
)

sns.boxplot(
    data=eda_metadata,
    x="duration",
    y="class",
    order=median_order,
    hue="class",
    palette="crest",
    ax=axes[1],
    legend=False  # disables legend as all classes already labeled on y-axis
)
axes[1].set_title("Duration by Class")
axes[1].set_xlabel("Duration, seconds")
axes[1].set_ylabel("Class")

plt.tight_layout()
plt.show()

### Заметки о длительности
 
Большинство клипов имеют относительно узкий диапазон длительности. Это облегчает использование общей схемы предобработки, хотя обрезка до фиксированной длины всё же требует проверки на клипы с короткими транзиентными событиями.

In [ ]:
split_class_share = (
    eda_metadata.groupby(["split", "class"])
    .size()
    .rename("num_samples")
    .reset_index()
)

split_duration = eda_metadata.groupby("split")["duration"].agg(["mean", "median", "std"]).reset_index()
display(split_duration)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=split_class_share, x="class", y="num_samples", hue="split", palette="Set1", ax=axes[0])
axes[0].set_title("Class Distribution by Split")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of samples")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(title="Split")

sns.violinplot(data=eda_metadata, x="split", y="duration", hue="split", palette="pastel", inner="quartile", ax=axes[1], legend=False)
axes[1].set_title("Duration Distribution by Split")
axes[1].set_xlabel("Split")
axes[1].set_ylabel("Duration, seconds")

plt.tight_layout()
plt.show()

### Выводы по датасету

UrbanSound8K подходит для задачи классификации городских звуков в формате супервизии без необходимости сильного глобального ребалансирования классов.
 
Основным ограничением при оценке результатов является протокол разбиения на фолды, а не общий размер датасета.
 
Статистика длительности выглядит достаточно стабильной для единообразной схемы извлечения признаков, однако за миноритарными классами требуется особенно внимательно следить при оценке модели.